In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
from astropy.time import Time
from astropy import units as u
from astropy.cosmology import Planck18 as cosmo
from astropy.cosmology import z_at_value
np.random.seed(42)

In [ ]:
injections = pd.read_csv('injections_cleaned.csv')
allsky = pd.read_csv('allsky_cleaned.csv')

In [ ]:
allsky = pd.read_csv('allsky_cleaned.csv')
injections_all = pd.read_csv('injections_cleaned.csv')

# Drop invalid rows (text export often adds one trailing NaN row).
valid = injections_all[['mass1', 'mass2', 'spin1z', 'spin2z', 'distance']].notna().all(axis=1)
injections_all = injections_all.loc[valid].copy().reset_index(drop=True)

# bayestar-inject stores detector-frame masses in injections.dat.
# Convert to source-frame for ejecta calculations.
redshift_all = np.array([
    z_at_value(cosmo.luminosity_distance, d * u.Mpc).value
    for d in injections_all['distance'].values
])
injections_all['redshift'] = np.around(redshift_all, decimals=7)

# Switch mass1 and mass2 to make sure mass1 >= mass2
swap = injections_all['mass1'] < injections_all['mass2']
injections_all.loc[swap, ['mass1', 'mass2']] = injections_all.loc[swap, ['mass2', 'mass1']].values
injections_all.loc[swap, ['spin1z', 'spin2z']] = injections_all.loc[swap, ['spin2z', 'spin1z']].values

injections_all['mass1_detector'] = injections_all['mass1'].values
injections_all['mass2_detector'] = injections_all['mass2'].values
injections_all['mass1_source'] = np.around(injections_all['mass1_detector'] / (1 + redshift_all), decimals=7)
injections_all['mass2_source'] = np.around(injections_all['mass2_detector'] / (1 + redshift_all), decimals=7)

# BNS selection should be done in source frame.
bns_index = np.logical_and(
    injections_all['mass1_source'] < 2.0606,
    injections_all['mass2_source'] < 2.0606,
)
injections = injections_all.loc[bns_index].copy()
allsky = allsky.loc[bns_index].copy().reset_index(drop=True)

print(f'Total valid injections: {len(injections_all)}')
print(f'BNS injections (source-frame cut): {len(injections)}')
print(f"mass2_detector max: {injections['mass2_detector'].max():.4f}")
print(f"mass2_source   max: {injections['mass2_source'].max():.4f}")
injections[['mass1_detector','mass2_detector','mass1_source','mass2_source','redshift']].head()


Using SFHo EOS to calculate compactness C. The grid data is saved at 'eos.mr'.

In [ ]:
eos = pd.read_csv('eos.mr', names=['radius', 'mass'], sep='\s+', skiprows=1)

mass = eos['mass'].values
radius = eos['radius'].values
f = interp1d(mass, radius, kind='cubic', fill_value="extrapolate")
mass_int = np.linspace(mass.min(), mass.max(), 1000)
radius1 = f(mass_int)

plt.figure(figsize=(8,6))
plt.scatter(mass, radius, label='EOS Data', color='b',s=3)
plt.plot(mass_int, radius1, label='Interpolated', color='r', alpha=0.5, linewidth=5)
plt.xlabel('Mass (M_sun)')
plt.ylabel('Radius (km)')
plt.title('Mass-Radius Relation from EOS')
plt.legend()
plt.grid()

In [ ]:
def compute_radius(mass, eos_file):
    eos = pd.read_csv(eos_file, names=['radius', 'mass'], sep='\s+', skiprows=1)
    mass_eos = eos['mass'].values
    radius_eos = eos['radius'].values
    f = interp1d(mass_eos, radius_eos, kind='cubic', fill_value="extrapolate")
    radius = f(mass)
    return radius

In [ ]:
mass1 = injections['mass1'].values
mass2 = injections['mass2'].values

radius1 = np.around(f(mass1),decimals=6)
radius2 = np.around(f(mass2),decimals=6)
injections['radius1'] = radius1
injections['radius2'] = radius2

In [ ]:
def compute_compactness(mass, radius):

    G = 6.67430e-11  # m^3 kg^-1 s^-2
    c = 299792458    # m/s
    Ms = 1.98855e30  # kg
    C = (G * mass * Ms) / (c**2 * radius * 1000)  # radius in km to m
    return np.around(C, decimals=5)

compactness1 = compute_compactness(mass1, radius1)
compactness2 = compute_compactness(mass2, radius2)
injections['compactness1'] = compactness1
injections['compactness2'] = compactness2

## Compute ejecta mass
In Bulla KN model, ejecta has two components: dynamical ejecta and wind ejecta. We can calculate them with gravitational wave parameters. 

Reference: *Predictions for electromagnetic counterparts to Neutron Star mergers discovered during LIGO-Virgo-KAGRA observing runs 4 and 5*

For dynamical ejecta mass, 
$log_{10}(m^{dyn fit}_{ej})=[a\frac{(1-2C_1)M_1}{C_1}+bM_2(\frac{M_1}{M_2})^n+\frac{d}{2}]+[1\leftrightarrow 2]$
, where $a=-0.0719, b=0.2116, d=-2.42, n=-2.905$;

$C=\frac{GW}{c^2R}$

In [ ]:
def dynamical_ejecta_mass(m1, m2, c1, c2):
    a = -0.0719
    b = 0.2116
    d = -2.42
    n = -2.905
    
    term1 = (a * (1 - 2 * c1) * m1 / c1) + (b * m2 * (m1 / m2) ** n) + (d / 2)
    term2 = (a * (1 - 2 * c2) * m2 / c2) + (b * m1 * (m2 / m1) ** n) + (d / 2)
    
    log_mej_dyn = term1 + term2
    mej_dyn = 10 ** log_mej_dyn
    
    return mej_dyn

In [ ]:
mej_dyn_fit = dynamical_ejecta_mass(mass1, mass2, compactness1, compactness2)
mej_dyn = np.minimum(0.09, mej_dyn_fit)
injections['mej_dyn'] = np.around(mej_dyn, decimals=6)
print("Dynamical Ejecta Mass(M_sun):")
print(mej_dyn)

For wind ejcta mass:

$m_{ej}^{wind}=\zeta m_{disc}$, $\zeta$ is sampled uniformly from $[0.1,0.4]$;

$log_{10}(m_{disk})=max(-3,a(1+b \cdot \tanh[\frac{c-(M_1+M_2)/M_{thr}}{d}]))$,where $a=-31.335,b=-0.9760,c=1.0474,d=0.05957$;

$M_{thr} = (2.38-3.606\frac{M_{TOV}}{R_{1.6M_{\odot}}})M_{TOV}$,$M_{TOV}=2.05M_{\odot}$

In [ ]:
def compute_wind_ejecta_mass(m1, m2, Mthr, zeta):
    a = -31.335
    b = -0.9760
    c = 1.0474
    d = 0.05957
    
    Mtot = m1 + m2
    log_mdisk = np.maximum(-3, a * (1 + b * np.tanh((c - Mtot / Mthr) / d)))
    mdisk = 10 ** log_mdisk
    
    mej_wind = zeta * mdisk
    return mej_wind

In [ ]:
Rthr = f(1.6)
MTOV = 2.05  # in solar masses
Mthr = (2.38 - 3.606 * (MTOV / Rthr)) * MTOV
print("Radius at 1.6M_sun R_1.6(km):", Rthr)
print("Threshold Mass M_thr(M_sun):", Mthr)

In [ ]:
zeta = np.random.uniform(0.1, 0.4, size=len(mass1))
mej_wind = compute_wind_ejecta_mass(mass1, mass2, Mthr, zeta)
injections['mej_wind'] = np.around(mej_wind, decimals=6)
print("Wind Ejecta Mass(M_sun):")
print(mej_wind)

If $m_1+m_2 \geq M_{thr}$, set both dynamical and wind ejcta to zero.

In [ ]:
index = np.where((mass1 + mass2) >= Mthr)[0]
print("Indices with total mass >= M_thr:", index)
injections.loc[index, 'mej_dyn'] = 0.0
injections.loc[index, 'mej_wind'] = 0.0

Save new injections parameters.

In [ ]:
# Heatmaps of ejecta masses vs (mass1, mass2)
bins = 25

# 2D mean for dynamical ejecta
H_sum_dyn, xedges, yedges = np.histogram2d(mass1, mass2, bins=bins, weights=mej_dyn)
H_count, _, _ = np.histogram2d(mass1, mass2, bins=[xedges, yedges])
H_mean_dyn = H_sum_dyn / np.where(H_count == 0, np.nan, H_count)

# 2D mean for wind ejecta
H_sum_wind, _, _ = np.histogram2d(mass1, mass2, bins=[xedges, yedges], weights=mej_wind)
H_mean_wind = H_sum_wind / np.where(H_count == 0, np.nan, H_count)

fig, axs = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)

im0 = axs[0].pcolormesh(xedges, yedges, H_mean_dyn.T, cmap='viridis', shading='auto')
axs[0].set_xlabel('mass1 (M_sun)')
axs[0].set_ylabel('mass2 (M_sun)')
axs[0].set_title('Mean Dynamical Ejecta Mass')
c0 = fig.colorbar(im0, ax=axs[0])
c0.set_label('mej_dyn (M_sun)')

im1 = axs[1].pcolormesh(xedges, yedges, H_mean_wind.T, cmap='viridis', shading='auto')
axs[1].set_xlabel('mass1 (M_sun)')
axs[1].set_ylabel('mass2 (M_sun)')
axs[1].set_title('Mean Wind Ejecta Mass')
c1 = fig.colorbar(im1, ax=axs[1])
c1.set_label('mej_wind (M_sun)')

plt.show()

In [ ]:
# total mass of ejecta
mej_total = mej_dyn + mej_wind
injections['mej_total'] = np.around(mej_total, decimals=6)

plt.hist(injections['mej_total'], bins=25, color='purple', alpha=0.7)
plt.xlabel('Total Ejecta Mass (M_sun)')
plt.ylabel('Number of Injections')
plt.title('Histogram of Total Ejecta Mass')
plt.show()

In [ ]:
# 2D mean (and count) of total ejecta mass vs (mass1, mass2)
bins = 25

H_sum, xedges, yedges = np.histogram2d(mass1, mass2, bins=bins, weights=mej_total)
H_count, _, _ = np.histogram2d(mass1, mass2, bins=[xedges, yedges])
H_mean = H_sum / np.where(H_count == 0, np.nan, H_count)

fig, axs = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)

im0 = axs[0].pcolormesh(xedges, yedges, H_mean.T, cmap='inferno', shading='auto')
axs[0].set_xlabel('mass1 (M_sun)')
axs[0].set_ylabel('mass2 (M_sun)')
axs[0].set_title('Mean Total Ejecta Mass (mej_total)')
c0 = fig.colorbar(im0, ax=axs[0])
c0.set_label('mean mej_total (M_sun)')

# show counts in the other panel
im1 = axs[1].pcolormesh(xedges, yedges, H_count.T, cmap='viridis', shading='auto')
axs[1].set_xlabel('mass1 (M_sun)')
axs[1].set_ylabel('mass2 (M_sun)')
axs[1].set_title('Number of Injections per Bin')
c1 = fig.colorbar(im1, ax=axs[1])
c1.set_label('counts')

plt.show()

Add half-opening angle($\Phi$) and observing angle($Cos\Theta$).

$\Phi \in U(15^o, 75^o)$, $\Theta=min(\iota, \pi-\iota)$ 

In [ ]:
# half-opening angle phi
phi = np.random.uniform(15, 75, size=len(mass1))

# observing angle theta
inclination = injections['inclination'].values
theta = np.where(inclination <= np.pi/2, inclination, (np.pi - inclination))

plt.figure(figsize=(16,6))
plt.subplot(1,2,1)
plt.hist(phi, bins=30, color='c', edgecolor='k', alpha=0.7)
plt.xlabel('Half-opening angle φ (degrees)')
plt.ylabel('Number of injections')
plt.title('Distribution of Half-opening Angle φ')
plt.grid()
plt.subplot(1,2,2)
plt.hist(theta, bins=30, color='m', edgecolor='k', alpha=0.7)
plt.xlabel('Observing angle θ (degrees)')
plt.ylabel('Number of injections')
plt.title('Distribution of Observing Angle θ')
plt.grid()
plt.tight_layout()

injections['phi'] = np.around(phi, decimals=6)
injections['costheta'] = np.around(np.cos(theta), decimals=6)

### Add distance and uncertainty in skymap.

In [ ]:
ra = np.rad2deg(injections['longitude'].values)
dec = np.rad2deg(injections['latitude'].values)

injections['ra'] = np.around(ra, decimals=6)
injections['dec'] = np.around(dec, decimals=6)

In [ ]:
distmean = allsky['distmean'].values
diststd = allsky['diststd'].values

injections['distmean'] = np.around(distmean, decimals=6)
injections['diststd'] = np.around(diststd, decimals=6)

In [ ]:
# add GPS time and mjd time columns
mjd_min = 61000.0
mjd_max = 64500.0
mjd_times = np.random.uniform(mjd_min, mjd_max, size=len(injections))
gps_times = Time(mjd_times, format='mjd').gps

injections['mjd_time'] = np.around(mjd_times, decimals=4)
injections['gps_time'] = np.around(gps_times, decimals=2)

In [ ]:
# re-order columns
cols = list(injections.columns)
cols.insert(1, cols.pop(cols.index('mjd_time')))
cols.insert(2, cols.pop(cols.index('gps_time')))
cols.insert(5, cols.pop(cols.index('ra')))
cols.insert(6, cols.pop(cols.index('dec')))

injections = injections[cols]
print(injections.columns)

In [ ]:
injections

In [ ]:
injections.to_csv('injections_final.csv', index=False)